In [1]:
import numpy as np
import pandas as pd
from scipy.integrate import solve_ivp
from multiprocessing import Pool, cpu_count

# ================================
# 1. PARAMETER SAMPLING
# ================================
def sample_parameters():
    return {
        "k1": np.random.uniform(0.1, 1.0),
        "km1": np.random.uniform(0.1, 1.0),
        "kdeg": np.random.uniform(0.01, 0.2),

        "kin": np.random.uniform(0.1, 1.0),
        "kout": np.random.uniform(0.1, 1.0),

        "k2": np.random.uniform(0.1, 1.0),
        "km2": np.random.uniform(0.1, 1.0),
        "kdp1": np.random.uniform(0.01, 0.2),

        "k3": np.random.uniform(0.1, 1.0),
        "km3": np.random.uniform(0.1, 1.0),
        "kdp2": np.random.uniform(0.01, 0.2),

        "k4": np.random.uniform(0.1, 1.0),
        "km4": np.random.uniform(0.1, 1.0),
        "kdp3": np.random.uniform(0.01, 0.2),

        "kprolif": np.random.uniform(0.1, 0.5),
        "kdecay": np.random.uniform(0.01, 0.1),
    }

# ================================
# 2. ODE SYSTEM
# ================================
def ode_system(t, y, p):

    ATP, P2X7, P2X7s, Ca_cyt, Ca_ext, MAP3K, MAP3Ks, JNK, JNKs, cJun, cJuns, PTC = y

    # 1. ATP + P2X7
    dP2X7s = p["k1"]*ATP*P2X7 - p["km1"]*P2X7s
    dP2X7  = -p["k1"]*ATP*P2X7 + p["km1"]*P2X7s
    dATP   = -p["k1"]*ATP*P2X7 + p["km1"]*P2X7s - p["kdeg"]*ATP

    # 2. Ca2+
    dCa_cyt = p["kin"]*P2X7s*(Ca_ext - Ca_cyt) - p["kout"]*Ca_cyt \
              - p["k2"]*Ca_cyt*MAP3K + p["km2"]*MAP3Ks

    dCa_ext = -p["kin"]*P2X7s*(Ca_ext - Ca_cyt)

    # 3. MAP3K
    dMAP3Ks = p["k2"]*Ca_cyt*MAP3K - p["km2"]*MAP3Ks - p["kdp1"]*MAP3Ks \
              - p["k3"]*MAP3Ks*JNK + p["km3"]*JNKs

    dMAP3K  = -p["k2"]*Ca_cyt*MAP3K + p["km2"]*MAP3Ks + p["kdp1"]*MAP3Ks

    # 4. JNK
    dJNKs = p["k3"]*MAP3Ks*JNK - p["km3"]*JNKs - p["kdp2"]*JNKs \
            - p["k4"]*JNKs*cJun + p["km4"]*cJuns

    dJNK  = -p["k3"]*MAP3Ks*JNK + p["km3"]*JNKs + p["kdp2"]*JNKs

    # 5. cJun
    dcJuns = p["k4"]*JNKs*cJun - p["km4"]*cJuns - p["kdp3"]*cJuns \
             - p["kprolif"]*cJuns + p["kdecay"]*PTC

    dcJun  = -p["k4"]*JNKs*cJun + p["km4"]*cJuns + p["kdp3"]*cJuns

    # 6. Tumor cells
    dPTC = p["kprolif"]*cJuns - p["kdecay"]*PTC

    return [
        dATP, dP2X7, dP2X7s,
        dCa_cyt, dCa_ext,
        dMAP3K, dMAP3Ks,
        dJNK, dJNKs,
        dcJun, dcJuns,
        dPTC
    ]

# ================================
# 3. SINGLE SIMULATION
# ================================
def run_simulation(sim_id):

    params = sample_parameters()

    # Initial conditions
    y0 = [
        np.random.uniform(5, 15),  # ATP
        np.random.uniform(2, 10),  # P2X7
        0,
        0.1,
        np.random.uniform(1, 5),
        np.random.uniform(2, 10),
        0,
        np.random.uniform(2, 10),
        0,
        np.random.uniform(2, 10),
        0,
        0
    ]

    t_eval = np.linspace(0, 100, 200)

    sol = solve_ivp(
        lambda t, y: ode_system(t, y, params),
        (0, 100),
        y0,
        t_eval=t_eval,
        method="RK45"
    )

    data = pd.DataFrame(sol.y.T, columns=[
        "ATP","P2X7","P2X7_star","Ca_cyt","Ca_ext",
        "MAP3K","MAP3K_star","JNK","JNK_star",
        "cJun","cJun_star","PTC"
    ])

    # Add metadata
    data["time"] = sol.t
    data["simulation_id"] = sim_id

    # ================================
    # 4. ADD PARAMETERS (CRITICAL FIX)
    # ================================
    for key, value in params.items():
        data[key] = value

    # ================================
    # 5. ADD BIOLOGICAL NOISE
    # ================================
    numeric_cols = [
        "ATP","P2X7","P2X7_star","Ca_cyt","Ca_ext",
        "MAP3K","MAP3K_star","JNK","JNK_star",
        "cJun","cJun_star","PTC"
    ]

    noise = np.random.normal(0, 0.05, size=data[numeric_cols].shape)
    data[numeric_cols] += noise * data[numeric_cols]

    return data

# ================================
# 6. PARALLEL DATA GENERATION
# ================================
def generate_dataset(n_simulations=100):

    print(f"Running {n_simulations} simulations on {cpu_count()} cores...")

    with Pool(cpu_count()) as pool:
        results = pool.map(run_simulation, range(n_simulations))

    dataset = pd.concat(results, ignore_index=True)

    return dataset

# ================================
# 7. MAIN
# ================================
if __name__ == "__main__":

    N_SIMULATIONS = 100  # increase to scale

    dataset = generate_dataset(N_SIMULATIONS)

    print("Saving dataset to Parquet...")

    dataset.to_parquet("update synthetic_ode_dataset.parquet")

    print("Done!")
    print(dataset.head())

Running 100 simulations on 2 cores...
Saving dataset to Parquet...
Done!
         ATP      P2X7  P2X7_star    Ca_cyt    Ca_ext     MAP3K  MAP3K_star  \
0  10.581866  5.182254   0.000000  0.101441  4.118813  7.289764    0.000000   
1   5.362193  0.476805   4.927693  0.690010  2.837667  7.293680    0.461103   
2   4.256343  0.246135   4.367720  0.575327  2.143917  6.681858    0.861044   
3   4.078070  0.256342   5.423772  0.519667  1.562128  5.519551    0.847458   
4   3.980308  0.262532   5.033075  0.435662  1.012296  5.135112    0.763160   

        JNK  JNK_star      cJun  ...       km2      kdp1        k3       km3  \
0  5.667005  0.000000  8.491446  ...  0.339719  0.159361  0.304425  0.377667   
1  5.555481  0.088430  8.179061  ...  0.339719  0.159361  0.304425  0.377667   
2  5.080128  0.254703  7.640502  ...  0.339719  0.159361  0.304425  0.377667   
3  4.426965  0.344434  7.254749  ...  0.339719  0.159361  0.304425  0.377667   
4  3.448229  0.371780  7.708723  ...  0.339719  0.15